# AI-OBEP — Quickstart Google Colab

Notebook ini menyiapkan seluruh repo AI-OBEP di Colab dalam beberapa menit:
clone repo → install paket → cek instalasi → siapkan data → demo singkat model ML.

Jalankan sel **dari atas ke bawah** (`Runtime → Run all` juga boleh). Tidak perlu GPU.

> Catatan: file di Colab **hilang** saat runtime diputus. Lihat sel terakhir untuk menyimpan hasil ke Google Drive.

## 1. Clone repo

In [ ]:
import os
REPO_URL = "https://github.com/ikigai-arch/AI-OBEP.git"
REPO_DIR = "/content/AI-OBEP"

if not os.path.exists(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only
%cd {REPO_DIR}
!ls

## 2. Install paket
Colab sudah punya numpy/pandas/sklearn/matplotlib; sel ini menambah pymoo, pythermalcomfort, xgboost, lightgbm, shap. ±1–2 menit.

In [ ]:
!pip install -q -r requirements-all.txt

## 3. Cek instalasi + ekstrak data proyek root
Semua baris paket harus `[ OK ]`.

In [ ]:
!python check_setup.py --fix-data

## 4. Data untuk `AI-OBEP-MLPaper` (diunduh dari GitHub ROBOD, ±20 MB)

In [ ]:
!bash AI-OBEP-MLPaper/scripts/00_get_data.sh

## 5. Demo cepat: surrogate + NSGA-II (±1 menit)

Memakai model surrogate yang sudah dilatih (`AI-OBEP-MLPaper/outputs/models/surrogate.joblib`),
menghitung energi & kenyamanan layout awal, lalu menjalankan NSGA-II singkat (3.000 evaluasi; paper: 25.000).

In [ ]:
import sys, numpy as np
sys.path.insert(0, f"{REPO_DIR}/AI-OBEP-MLPaper/src")
from obep_mlpaper.surrogate import Surrogate
from obep_mlpaper.layout import case_study_1, evaluate
from obep_mlpaper.nsga import PAPER_HP, run_nsga2

sur = Surrogate.load(f"{REPO_DIR}/AI-OBEP-MLPaper/outputs/models/surrogate.joblib").single_thread()
sc = case_study_1()
base = evaluate(sc, sur, sc.baseline()[None])
print(f"Layout awal @24 °C : energi = {base['F'][0,0]:.2f} kW, rata-rata |PMV| = {base['F'][0,1]:.3f}")

res = run_nsga2(sc, sur, PAPER_HP[sc.name], seed=1, n_evals=3000)
F = res["F"]
i_e, i_c = F[:, 0].argmin(), F[:, 1].argmin()
print(f"Jumlah solusi Pareto: {len(F)}")
print(f"Solusi hemat energi : {F[i_e,0]:.2f} kW ({1 - F[i_e,0]/base['F'][0,0]:.1%} lebih hemat), |PMV| {F[i_e,1]:.3f}")
print(f"Solusi paling nyaman: {F[i_c,0]:.2f} kW, |PMV| {F[i_c,1]:.3f}")

In [ ]:
import matplotlib.pyplot as plt
plt.scatter(F[:, 0], F[:, 1], label="Front Pareto NSGA-II")
plt.scatter(*base["F"][0], marker="*", s=200, c="red", label="Layout awal @24 °C")
plt.xlabel("Energi HVAC f_cost [kW]"); plt.ylabel("Rata-rata |PMV| f_tc"); plt.legend(); plt.grid(alpha=.3)
plt.title("Trade-off energi vs kenyamanan (kasus 1)"); plt.show()

## 6. Menjalankan notebook lengkap

**Cara A — eksekusi otomatis di sini** (output tersimpan di dalam file notebook-nya):

In [ ]:
# Proyek root (EDA -> surrogate -> optimasi). Notebook 03 bisa lama; set QUICK=True di dalamnya untuk uji cepat.
!cd notebooks && jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=-1 01_explore_dataset.ipynb
# !cd notebooks && jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=-1 02_train_surrogate.ipynb

**Cara B — buka notebook secara interaktif:** di Colab pilih `File → Open notebook → GitHub`, tempel
`ikigai-arch/AI-OBEP`, pilih notebooknya, lalu **tambahkan sel ini paling atas** sebelum sel lain:

```python
import os
if not os.path.exists("/content/AI-OBEP"):
    !git clone --depth 1 https://github.com/ikigai-arch/AI-OBEP.git /content/AI-OBEP
    !pip install -q -r /content/AI-OBEP/requirements-all.txt
%cd /content/AI-OBEP/notebooks          # untuk notebook AI-OBEP-MLPaper: /content/AI-OBEP/AI-OBEP-MLPaper/notebooks
```

## 7. (Opsional) Simpan hasil ke Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/AI-OBEP_outputs
!cp -r outputs AI-OBEP-MLPaper/outputs /content/drive/MyDrive/AI-OBEP_outputs/
print("Tersimpan di MyDrive/AI-OBEP_outputs")